# Baseline preparation and ledger rehearsal

Build five targets from saved fixtures, inspect abstentions and snapshot identity.
The optional Postgres section creates and removes a local disposable database.
Use the Baseline Python 3.12 kernel; offline cells take seconds, Postgres about 15 seconds.
All inputs are invented; these outputs are not official forecasts.


In [ ]:
import json
import tempfile
from pathlib import Path

import pandas as pd

from hoop.domains.nba.baselines.model import prepare
from hoop.platform.prepared import save_local
from hoop.platform.provenance import parse_time

ROOT = Path.cwd()
inputs = json.loads((ROOT / "tests/fixtures/nba/preparation.json").read_text())

## Run and inspect

The roster includes a player without history. Every roster member stays in the denominator. Minutes are averaged first; pooled rates use that same appearance window. Availability is descriptive in this baseline.


In [ ]:
result = prepare(inputs, generated_at=parse_time(inputs["fixture_generated_at"]))
display(pd.DataFrame(result["entries"])[["subject_id", "target", "value", "unit", "reason_code"]])
print("Prepared coverage:", result["coverage"])

In [ ]:
(ROOT / ".cache").mkdir(exist_ok=True)
with tempfile.TemporaryDirectory(dir=ROOT / ".cache") as folder:
    saved = save_local(result, Path(folder))
    replay = save_local(result, Path(folder))
    print("Snapshot:", saved["snapshot_id"], "| replay reused:", replay["reused"])

## Interpretation

Player 201 has 10 and 30 prior minutes, yielding 20 predicted minutes and 12 points. The rookie has five explicit abstentions. With complete common windows, this minutes/rate baseline equals each counting stat's recent mean; it is not evidence of predictive improvement. Saved inputs, code and configuration identify the prepared values. No database, live API, model activation or official publication runs here.


## Protected publication

SQL binds artifact bytes, both saved roster memberships and the active bundle. A second transaction observes the committed batch before the cutoff. Its receipt time is an upper bound on the first commit, not an exact commit timestamp.


In [ ]:
RUN_POSTGRES_REHEARSAL = False  # Opt in only after configuring local BASELINE_TEST_DSN.

In [ ]:
if RUN_POSTGRES_REHEARSAL:
    from scripts.rehearse_publication import run

    publication_report = run()
    display(pd.DataFrame([publication_report]))
else:
    publication_report = {"state": "not_run", "reason": "local Postgres opt-in required"}
    print(publication_report)

## Receipt interpretation

The optional run uses the same protected client as production and verifies an exact retry. Integration tests also cover rollback, lost insertion/receipt responses, late commits and invalidation. A batch without a timely receipt remains stored and unverified; recovery confirms that same identity. Production requires separate hosted/source/backup acceptance. Outcomes follow in OUTCOME-001.


## Final outcomes: confirmed DNP versus unknown

Use invented complete finals, with 53 minutes and 12.50 seconds to preserve overtime precision. The saved cohort comes from the earlier prediction inputs. Missing rows remain unknown; source-confirmed nonparticipation can resolve zeros. These fixtures do not establish live source semantics.


In [ ]:
from datetime import UTC, datetime

from hoop.domains.nba.outcomes import normalize_final, outcome_entries
from tests.fixtures.finals import final_context, final_payload

final_body = final_payload()
parsed_final = normalize_final(final_body, final_context(), datetime(2026, 10, 27, tzinfo=UTC))
outcomes_partial = outcome_entries(parsed_final, result["cohort"])
display(pd.DataFrame(outcomes_partial)[["subject_id", "target", "value", "reason_code"]])

In [ ]:
final_body["game"]["homeTeam"]["players"].append(
    {"personId": 999, "played": "0", "notPlayingReason": "inactive"}
)
corrected_final = normalize_final(final_body, final_context(), datetime(2026, 10, 27, tzinfo=UTC))
outcomes_corrected = outcome_entries(corrected_final, result["cohort"])
comparison = (
    pd.DataFrame(result["entries"])[["subject_id", "target", "value"]]
    .rename(columns={"value": "prediction"})
    .merge(
        pd.DataFrame(outcomes_corrected)[["subject_id", "target", "value"]].rename(
            columns={"value": "actual"}
        ),
        on=["subject_id", "target"],
    )
)
display(comparison)

In [ ]:
if RUN_POSTGRES_REHEARSAL:
    outcome_report = run(include_outcomes=True)
    display(pd.DataFrame([outcome_report]))
else:
    outcome_report = {"state": "not_run", "reason": "local Postgres opt-in required"}

## Outcome revisions and timing

The optional rehearsal stores partial and corrected versions while preserving every prediction and receipt. Its compressed timing scenario deliberately supplies a final with an earlier start, which invalidates the fixture forecast; it cannot become valid accuracy evidence. The first publication rehearsal remains separate. Live source acceptance is still pending. Accuracy aggregation belongs to EVAL-001.
